In [ ]:
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_google_genai import ChatGoogleGenerativeAI

load_dotenv(override=True)

In [ ]:
open_router_api_key = os.environ.get("OPEN_ROUTER_API_KEY")
if not open_router_api_key:
    raise ValueError("OPEN_ROUTER_API_KEY is not set in the environment variables.")

gemini_api_key = os.environ.get("GEMINI_API_KEY")
if not gemini_api_key:
    raise ValueError("GEMINI_API_KEY is not set in the environment variables.")

In [ ]:
def call_llm(model, messages):
    if model == "gpt-4":
        llm = ChatOpenAI(model_name="openrouter/free",
            api_key=open_router_api_key,
            base_url="https://openrouter.ai/api/v1",
            temperature=0.5)
    elif model == "gemini-pro":
        llm = ChatGoogleGenerativeAI(model="gemini-3.5-flash", temperature=0)
    else:
        raise ValueError(f"Unsupported model: {model}")

    response = llm.invoke(messages)
    return response

In [ ]:
SYSTEM_PROMPT = "You are a helpful assistant."
USER_PROMPT = "What is the capital of France?"

messages = [
    {"role": "system", "content": SYSTEM_PROMPT},
    {"role": "user", "content": USER_PROMPT}
]

In [ ]:
response = call_llm("gemini-pro", messages)

In [ ]:
print(response.content)

In [ ]:
from typing import TypedDict

class AgentState(TypedDict):
    question: str
    response: str
    model_name: str

In [ ]:
route_llm = ChatOpenAI(model_name="openrouter/free",
            api_key=open_router_api_key,
            base_url="https://openrouter.ai/api/v1",
            temperature=0.5)

In [ ]:
SYSTEM_PROMPT = """
            You are a model routing assistant.

            Choose the most appropriate model for the user's question.

            Available models:

            1. fast_model
            - Use for simple factual questions and basic conversations.
            - Fast and inexpensive.

            2. powerful_model
            - Use for complex reasoning, analysis, and difficult questions.

            3. coding_model
            - Use for programming, debugging, algorithms, and software engineering questions.

            Return only the name of the selected model.
    """
USER_PROMPT = "What is the capital of France?"


In [ ]:

from pydantic import BaseModel, Field


class RoutingDecision(BaseModel):
    model_name: str = Field(
        description="The name of the selected model for routing the question."
    )

In [ ]:
route_llm_structured =  route_llm.with_structured_output(RoutingDecision) 

In [ ]:
fast_llm = ChatGoogleGenerativeAI(model="gemini-3.5-flash", temperature=0)

powerful_llm = ChatOpenAI(model_name="openrouter/free",
            api_key=open_router_api_key,
            base_url="https://openrouter.ai/api/v1",
            temperature=0.5)

coding_llm = ChatGoogleGenerativeAI(model="gemini-3.5-coding", temperature=0)

In [ ]:
models = {
    "fast_model": fast_llm,
    "powerful_model": powerful_llm,
    "coding_model": coding_llm
}

In [ ]:
def route_question(state: AgentState) -> AgentState:
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": state["question"]}
    ]

    response = route_llm_structured.invoke(messages)

    if response is None or not hasattr(response, 'model_name'):
        raise ValueError("Routing decision failed. No model name returned.")

    selected_model = response.model_name

    print(f"Selected model: {selected_model}")

    return {"selected_model": selected_model}


In [ ]:
def answer_question(state: AgentState) -> AgentState:
    selected_model = state["model_name"]

    print(f"Answering question using model: {selected_model}")

    if selected_model == "fast_model":
        llm = models["fast_model"]
    elif selected_model == "powerful_model":
        llm = models["powerful_model"]
    elif selected_model == "coding_model":
        llm = models["coding_model"]
    else:
        raise ValueError(f"Unsupported model: {selected_model}")

    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": state["question"]}
    ]

    response = llm.invoke(messages)

    return {"response": response.content, "model_name": selected_model}

In [ ]:

from langgraph.graph import StateGraph

graph = StateGraph(AgentState)

graph.add_node("route_question", route_question)
graph.add_node("answer_question", answer_question)

In [ ]:
from langgraph.graph import START, END

graph.add_edge(START, "route_question")
graph.add_edge("route_question", "answer_question")
graph.add_edge("answer_question", END)

app = graph.compile()

In [ ]:
result = app.invoke(
    {"question": "What is the capital of France?",
     "model_name": "",
     "response": ""}
)

In [ ]:
print(result)